# Tox Program TA/API Requirements

In [1]:
from math import sqrt, exp


# TODO: Implement standard study designs once we finalize those with LabCorp. Sort out how that will work, and what the data structure(s) need(s) to be.
# TODO: Figure out what parameters need to be reported to the user as assumptions.
# TODO: Model rabbit weight by age and sex. It's the only species we haven't modeled.


#### Variable(s): ###########################################

# animal body weight model - value of variable "k" comes from GraphPad Prism "Logistic Growth" fitting
# General equation is: average_study_body_weight = max_body_weight * min_body_weight / (min_body_weight + ((max_body_weight - min_body_weight) * e^(-k * study_duration)))
#                               bwmax       bwmin       k
bw_lut = {'sd_rat_male':        [1.2150,    0.2838,     0.01322],
          'sd_rat_female':      [0.6833,    0.1757,     0.00944],
          'sd_rat_mixed':       [0.9461,    0.2298,     0.01184],

          'wh_rat_male':        [0.5938,    0.3218,     0.00809],
          'wh_rat_female':      [0.3585,    0.1994,     0.00505],
          'wh_rat_mixed':       [0.4743,    0.2609,     0.00690],

          'mouse_male':         [0.0416,    0.0321,     0.01449],
          'mouse_female':       [0.0416,    0.0321,     0.01449],
          'mouse_mixed':        [0.0416,    0.0321,     0.01449],

        # New parameters (not in online tool) for large animal body weight calculation.
        # While the parameters above are for study duration, the parameters below give average weight by age.
          'cam_cyno_male':      [8.5740,    1.223,      0.3556],
          'cam_cyno_female':    [4.9450,    1.197,      0.3823],
          'cam_cyno_mixed':     [6.7595,    1.210,      0.3690],

          'bea_dog_male':       [11.74,     1.0670,     5.543],
          'bea_dog_female':     [8.820,     0.8175,     6.264],
          'bea_dog_mixed':      [10.28,     0.9423,     5.904],}

# # Parameters for common studies (not implemented yet)
# std_group_size = {
#     'glp_tox':
#         {'rat':     (32, 32, 42),
#          'dog':     (10, 10, 10),
#          'monkey':  (6,  6,  10),
#          'rabbit':  (20, 20, 20)},
#     'non-glp_tox':
#         {'rat':     (6,  6,  6),
#          'dog':     (3,  3,  3),
#          'monkey':  (6,  6,  10),
#          'rabbit':  (6,  6,  6)},
#     'glp_efd':
#         {'rabbit':  (20, 20, 20),
#          'rat':     (20, 20, 20),
#          'monkey':  (12, 12, 12)},
#     'non-glp_efd':
#         {'rabbit':  (6,  6,  6),
#          'rat':     (6,  6,  6)}
# }

#### Functions: #############################################
def calculate_ta_required(study_parameters):
    """
    Main function that calculates the required ta requirement for a study.
    :param study_parameters: dictionary with study parameters.
    :return: Prints the required TA amounts with overage, recommends number of DFAs.
    """
    if inputs_are_valid(study_parameters):

        gs =  study_parameters['group_size']
        dss = study_parameters['doses']
        sd =  study_parameters['study_duration']
        df =  study_parameters['dose_frequency']

        num_daily = [{'QD': 1, 'BID': 2}[f] for f in df]
        anim_wt = get_animal_weight(study_parameters)['animal_weight']
        group_dose_count = [gs * num * sd for gs, num in zip(gs, num_daily)]
        ta_req = [x[0] * x[1] * anim_wt for x in zip(dss, group_dose_count)]

        study_parameters['ta_required_g'] = [tr / 1000 for tr in ta_req]
        study_parameters['total_ta_required_g'] = sum(study_parameters['ta_required_g'])
        study_parameters['bulk_ta_required_g'] = [trg / study_parameters['fraction_active'] for trg in study_parameters['ta_required_g']]
        study_parameters['total_bulk_ta_required_g'] = sum(study_parameters['bulk_ta_required_g'])

        print_results(study_parameters)

    else:
        print(f"Please fix input(s) and try again.")


def inputs_are_valid(study_parameters):
    """
    Checks to see if inputs are valid.
    :param study_parameters: dictionary with study parameters
    :return: True or False
    """
    try:
        assert study_parameters['species'] in ('mouse', 'rat', 'dog', 'rabbit', 'monkey')
    except AssertionError:
        print("Species must be mouse, rat, rabbit, dog, or monkey.")
        return False
    try:
        assert type(study_parameters['group_size']) == tuple and sum(study_parameters['group_size']) > 0
    except AssertionError:
        print("Group size must be a positive non-empty tuple.")
        return False
    try:
        assert type(study_parameters['doses']) in (list, tuple)
    except AssertionError:
        print("Group size must be a list or tuple.")
        return False
    try:
        assert all([f in ('QD', 'BID') for f in study_parameters['dose_frequency']])
    except AssertionError:
        print("Dosing frequencies must be QD or BID.")
        return False
    try:
        assert type(study_parameters['study_duration']) == int
    except AssertionError:
        print("Duration must be a positive non-zero integer.")
        return False
    try:
        assert len(study_parameters['doses']) == len(study_parameters['dose_frequency'])
    except AssertionError:
        print("Must have the same number of doses and dosing frequencies.")
        return False
    try:
        assert study_parameters['sex'] in ('male', 'female', 'mixed')
    except AssertionError:
        print(f"Sex must be male, female or mixed.")
        return False
    try:
        assert study_parameters['experience'] in ('naive', 'non-naive')
    except AssertionError:
        print(f"Experience must be naive or non-naive.")
        return False
    # All inputs are valid
    return True


def plural(study_parameters):
    """
    For printing the output sentence.
    :param study_parameters: dictionary of study parameters
    :return: study_parameters with species plural
    """
    study_parameters['species_plural'] = {'mouse': 'mice', 'dog': 'dogs', 'rat': 'rats', 'rabbit': 'rabbits', 'monkey': 'monkeys'}[study_parameters['species']]
    return study_parameters


def study_type(study_parameters):
    """
    For printing output sentence.
    :param study_parameters: dictionary of study parameters
    :return: phrase for use in the output sentence
    """
    freq = study_parameters['dose_frequency']
    dur = study_parameters['study_duration']

    if not all([frq == freq[0] for frq in freq]):
        study_frequency = 'mixed frequency'
    else:
        study_frequency = freq[0]

    if dur != 1: return f"{dur}-day repeat {study_frequency} dosing"
    else: return "single dose"


def analysis_es(number):
    if number == 1: return 'analysis'
    else: return 'analyses'


def get_animal_weight(study_parameters):
    sp = study_parameters['species']
    sx = study_parameters['sex']
    sd = study_parameters['study_duration']
    ex = study_parameters['experience']

    if sp == 'rat':               sp = 'sd_rat'
    if sp == 'guinea pig':        wt = 0.45
    elif sp == 'mini pig':        wt = 18
    elif sp == 'rabbit':          wt = 4
    elif sp == 'dog':             wt = 12
    elif sp == 'marmoset':        wt = 0.5
    elif sp == 'cyno' or sp == 'monkey':
        if ex == 'naive':         wt = 7
        else:                     wt = 9

    else:
        bwmax, bwmin, k = bw_lut[f"{sp}_{sx}"]
        wt = bwmax * bwmin / (bwmin + ((bwmax - bwmin) * exp(-k * sd)))

    print(wt)
    study_parameters['animal_weight'] = wt

    return study_parameters


def print_results(study_parameters):
    api, bulk = study_parameters['total_ta_required_g'], study_parameters['total_bulk_ta_required_g']

    print(f'A {study_type(study_parameters)} {study_parameters['species']} study will require:')
    print(f'    Total Test Article (exact): {round(api, 1)} grams ({round(bulk, 1)} grams bulk)')
    print(f'    Total Test Article (+20%):  {round((api * 1.2), 1)} grams ({round(bulk * 1.2, 1)} grams bulk)')
    print(f'    Total Test Article (+30%):  {round((api * 1.3), 1)} grams ({round(bulk * 1.3, 1)} grams bulk)')
    print(f'    Total Animal Count: {sum(study_params['group_size'])} animals')

    num_formulations = {"BID": 2, "QD": 1, "QOD": 1/2, "QW": 1/7}[study_params['formulation_frequency']] * study_params['study_duration']
    num_dfas = int(round(sqrt(num_formulations), 0))
    if num_dfas == 0: num_dfas = 1

    print(f"\nIdeally, do {num_dfas} dose formulation {analysis_es(num_dfas)} (DFAs) in this study (square root rule).")

# Study Parameters

In [11]:
# Input Parameters:
study_params = {'study_duration':           14,                    # number of days in dosing period
                'species':                  'rat',               # mouse, rat, dog, monkey, rabbit
                'group_size':               (6, 6, 10),             # Number of animals in a group
                'doses':                    (10, 45, 100),          # list/tuple with dose levels in mg/kg
                'dose_frequency':           ('QD', 'QD', 'BID'),     # list/tuple with QD or BID for each dose in 'doses'
                'fraction_active':          0.25,                      # fraction of API in the TA
                'formulation_frequency':    'QD',                   # BID = twice daily, QD = daily, QOD = every other day, QW = weekly
                'sex':                      'male',                 # 'male', 'female', 'mixed'
                'experience':               'naive',                # Naive or non-naive

# Calculated Study Parameters:
                'species_plural':           'TBD',
                'animal_weight':            'TBD',
                'ta_required_g':            'TBD',
                'total_ta_required_g':      'TBD',
                'bulk_ta_required_g':       'TBD',
                'total_bulk_ta_required_g': 'TBD'}

calculate_ta_required(study_params)

0.32601801932692087
A 14-day repeat mixed frequency dosing rat study will require:
    Total Test Article (exact): 10.6 grams (42.5 grams bulk)
    Total Test Article (+20%):  12.8 grams (51.0 grams bulk)
    Total Test Article (+30%):  13.8 grams (55.3 grams bulk)
    Total Animal Count: 22 animals

Ideally, do 4 dose formulation analyses (DFAs) in this study (square root rule).


In [9]:
print("\n---------------------------------------")
print("dose      exact     +20%      +30%")
print(f"{'(mg/kg)':<10}{'(g)':<10}{'(g)':<10}{'(g)':<10}")
print("---------------------------------------")
for dose, ta_req in zip(study_params['doses'], study_params['ta_required_g']):
    ta_req_20 = ta_req * 1.2
    ta_req_30 = ta_req * 1.3
    print(f"{dose:<10}{round(ta_req, 1):<10}{round(ta_req_20, 1):<10}{round(ta_req_30, 1):<10}")
print("---------------------------------------")


---------------------------------------
dose      exact     +20%      +30%
(mg/kg)   (g)       (g)       (g)       
---------------------------------------
10        5.9       7.1       7.6       
45        26.5      31.8      34.4      
100       196.0     235.2     254.8     
---------------------------------------
